# Databricks Demo: one Delta table, batch vs. streaming

The same orders table is read two ways:
- **Batch** (`spark.read`): runs once and saves a snapshot.
- **Streaming** (`spark.readStream`): picks up new Delta commits incrementally, with a checkpoint so nothing is processed twice.
  - A Delta commit is one atomic, numbered change in the table’s log, and the table’s history is the list of its commits.
  - A Delta table is Parquet data files plus a transaction log (a _delta_log/ folder). Each write (an append, overwrite, update, delete or merge) adds one small numbered file to that log:

**Running on Databricks**
1. Workspace → Import → upload this `.ipynb`.
2. Attach the notebook to **Serverless** compute.

Free Edition is serverless-only, so the stream uses `trigger(availableNow=True)`: it processes everything new, then stops. The notebook reruns it to catch up after new orders arrive. 

The whole notebook can be rerun from the top: Cell 0 resets the demo tables.

## Cell 0 · Setup
Uses your workspace's current catalog, creates a schema and a volume for checkpoints, and clears anything left from a previous run.

In [0]:
import time
from pyspark.sql import functions as F

CATALOG = spark.sql("SELECT current_catalog()").first()[0]   # your default catalog
SCHEMA = "orders_demo"
print("Using catalog:", CATALOG)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.checkpoints")

orders_table   = f"{CATALOG}.{SCHEMA}.orders"
batch_table    = f"{CATALOG}.{SCHEMA}.revenue_snapshot"
stream_table   = f"{CATALOG}.{SCHEMA}.orders_stream"
checkpoint     = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints/orders_stream"
new_checkpoint = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints/orders_stream_new"   # used in Cell 11

# Start clean so the notebook can be rerun from the top
for q in spark.streams.active:
    q.stop()
for t in (orders_table, batch_table, stream_table):
    spark.sql(f"DROP TABLE IF EXISTS {t}")
for path in (checkpoint, new_checkpoint):
    try:
        dbutils.fs.rm(path, recurse=True)
    except Exception:
        pass   # nothing to remove on the first run

Using catalog: workspace


### Seed the source table with 6 orders
Each call to `add_orders` appends to the Delta table, which creates a new Delta commit. Streams pick up new commits.

In [0]:
ORDER_SCHEMA = "order_id INT, product STRING, amount DOUBLE"

def add_orders(rows):
    """Append orders to the source Delta table (one new Delta commit per call)."""
    spark.createDataFrame(rows, ORDER_SCHEMA).write.mode("append").saveAsTable(orders_table)

def snapshot_totals():
    """Totals from the saved batch snapshot."""
    return spark.table(batch_table).agg(F.sum("orders").alias("orders"), F.sum("revenue").alias("revenue"))

def stream_totals():
    """Totals from the rows the stream has written so far."""
    return spark.table(stream_table).agg(F.count("*").alias("orders"), F.sum("amount").alias("revenue"))

add_orders([
    (1, "laptop", 1200.0), (2, "monitor", 450.0), (3, "monitor", 450.0),
    (4, "headphones", 300.0), (5, "keyboard", 140.0), (6, "webcam", 400.0),
])
display(spark.table(orders_table).orderBy("order_id"))   # 6 orders, $2,940

order_id,product,amount
1,laptop,1200.0
2,monitor,450.0
3,monitor,450.0
4,headphones,300.0
5,keyboard,140.0
6,webcam,400.0


## Cells 1–3 · Batch: read once, save a snapshot

In [0]:
# Cell 1 · Read and aggregate
batch = spark.read.table(orders_table)

result = (batch.groupBy("product")
    .agg(F.count("*").alias("orders"),
         F.sum("amount").alias("revenue")))

display(result)

product,orders,revenue
laptop,1,1200.0
monitor,2,900.0
headphones,1,300.0
keyboard,1,140.0
webcam,1,400.0


In [0]:
# Cell 2 · Save the snapshot
# A DataFrame is lazy (just a query plan); writing it makes the snapshot.
result.write.mode("overwrite").saveAsTable(batch_table)

In [0]:
# Cell 3 · Snapshot totals
display(snapshot_totals())   # expected: 6 orders, $2,940

orders,revenue
6,2940.0


## Cells 4–6 · Streaming: same source, incremental rows
The stream copies order **rows** into `orders_stream`; revenue is computed by querying that table.

- **Serverless (Free Edition):** `availableNow` processes everything new, then stops. `catch_up()` simply runs it again.
- **Classic:** `processingTime="5 seconds"` keeps running; `catch_up()` just waits for the next micro-batch.

In [0]:
# Cell 4 · Define and start the stream
def run_stream(checkpoint_path=checkpoint):
    writer = (spark.readStream.table(orders_table)
        .writeStream
        .option("checkpointLocation", checkpoint_path))
    # Serverless: process everything new, then stop
    query = writer.trigger(availableNow=True).toTable(stream_table)
    query.awaitTermination()
    return query

query = run_stream()

In [0]:
# Cell 5 · Streaming output so far
display(stream_totals())   # expected: 6 orders, $2,940

orders,revenue
6,2940.0


In [0]:
# Cell 6 · Four new orders arrive
add_orders([(7, "laptop", 1200.0), (8, "keyboard", 140.0), (9, "headphones", 300.0), (10, "mouse", 70.0)])
run_stream()

print("Batch snapshot: unchanged until the batch is rerun")
display(snapshot_totals())   # expected: 6 orders, $2,940

print("Streaming output: 4 new rows processed")
display(stream_totals())     # expected: 10 orders, $4,650

Batch snapshot: unchanged until the batch is rerun


orders,revenue
6,2940.0


Streaming output: 4 new rows processed


orders,revenue
10,4650.0


## Cells 7–11 · Recovery: the checkpoint prevents duplicates

In [0]:
# Cell 7 · Stop the stream (on serverless it has already stopped by itself)
for q in spark.streams.active:
    q.stop()
print("Active streams:", len(spark.streams.active))
display(stream_totals())   # expected: still 10 rows

Active streams: 0


orders,revenue
10,4650.0


In [0]:
# Cell 8 · Restart with the SAME checkpoint
query = run_stream(checkpoint)
display(stream_totals())   # expected: still 10 rows, nothing processed twice

orders,revenue
10,4650.0


In [0]:
# Cell 9 · Append 2 more orders
add_orders([(11, "webcam", 400.0), (12, "mouse", 70.0)])
run_stream()
display(stream_totals())   # expected: 12 orders, $5,120

orders,revenue
12,5120.0


In [0]:
# Cell 10 · Rerun the batch: the snapshot catches up too
(spark.read.table(orders_table)
    .groupBy("product")
    .agg(F.count("*").alias("orders"), F.sum("amount").alias("revenue"))
    .write.mode("overwrite").saveAsTable(batch_table))

display(snapshot_totals())   # expected: 12 orders, $5,120

orders,revenue
12,5120.0


### Cell 11 · Counter-case: restart with a NEW checkpoint
A new checkpoint has no record of which Delta commits were already processed, so the stream starts from the beginning of the table and appends every order again.

In [0]:
for q in spark.streams.active:
    q.stop()

# A brand-new checkpoint has no record of what was already processed,
# so the stream reads all 12 orders from the start and appends them again.
run_stream(new_checkpoint)
display(stream_totals())   # expected: 24 rows, $10,240 (every order written twice)

# Which orders are duplicated?
display(spark.table(stream_table).groupBy("order_id").count().filter("count > 1").orderBy("order_id"))

orders,revenue
24,10240.0


order_id,count
1,2
2,2
3,2
4,2
5,2
6,2
7,2
8,2
9,2
10,2


## Cell 12 · Cleanup
Stop any running stream. On a classic cluster, a running stream keeps the cluster busy and billing, so also terminate the cluster when you're done.

In [0]:
for q in spark.streams.active:
    q.stop()
print("Active streams:", len(spark.streams.active))

# Optional: remove everything the demo created (tables, volume, checkpoints)
# spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA} CASCADE")

Active streams: 0


## Expected results

| Cell | Batch snapshot | Streaming output |
|---|---|---|
| 3 / 5 | 6 orders · $2,940 | 6 orders · $2,940 |
| 6 (after 4 new orders) | **6 orders · $2,940** (unchanged) | **10 orders · $4,650** |
| 8 (restart, same checkpoint) | — | 10 orders (no duplicates) |
| 9–10 (after 2 more orders) | 12 orders · $5,120 | 12 orders · $5,120 |
| 11 (new checkpoint) | — | 24 rows · $10,240 (duplicates) |

**Same DataFrame API. Different execution model.**